In [1]:
import pandas as pd

url = (
    "https://huggingface.co/datasets/"
    "datahiveai/Amazon-Reviews-Dataset/resolve/main/reviews.csv"
)

df = pd.read_csv(url, nrows=1000)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head(5))

Shape: (1000, 13)

Columns:
['id', 'asin', 'url', 'body', 'date', 'title', 'images', 'videos', 'stars', 'user_id', 'variation', 'found_helpful', 'verified_purchase']


,id,asin,url,body,date,title,images,videos,stars,user_id,variation,found_helpful,verified_purchase
0,RVAV3SWVU0O0W,B085DVHQ57,https://www.amazon.com/gp/customer-reviews/RVA...,Whenever asked by recent high school grads wha...,Sat Jul 05 2025,"My go-to water bottle, I take it everywhere I go",[],[],5,amzn1.account.AEPH7VYTQBYH6UW7INYUMMYNSBEA,Size: 24 Ounces Color: Shy Marshmallow,5,1
1,R1VVZ8F6WC4CWP,B085DVHQ57,https://www.amazon.com/gp/customer-reviews/R1V...,I bought the Owala FreeSip Insulated Stainless...,Fri Jun 06 2025,Perfect for Kids and Adults — My 12-Year-Old L...,[],[],5,amzn1.account.AE3WFQSNFH7ARSXPHUQZQV6CQ2TA,Size: 24 Ounces Color: Shy Marshmallow,47,1
2,R1N1YR132OT1B4,B085DVHQ57,https://www.amazon.com/gp/customer-reviews/R1N...,Can't say enough about this water bottle. I tr...,Thu Jun 26 2025,Not Even Close - Best H20 Bottle on the Market,[],[],5,amzn1.account.AE2PKPB3WT4L6WKMFUQHMA5ZMD3A,Size: 32 Ounces Color: Foggy Tide,25,1
3,R2NVM73BPETXF8,B085DVHQ57,https://www.amazon.com/gp/customer-reviews/R2N...,I've had this for over a year now and it's def...,Fri Jun 13 2025,Best water bottle,[],[],5,amzn1.account.AG4LH3S3WIYF4NQTZYUT2XHVAPMQ,Size: 24 Ounces Color: Foggy Tide,16,1
4,R20Q2RHQ3NGXB,B085DVHQ57,https://www.amazon.com/gp/customer-reviews/R20...,This is our third Owala Kids FreeSip bottle an...,Tue Jul 01 2025,"Great water bottle, easy clean",[],[],5,amzn1.account.AGZ43YBXQ6P5U4TGTIBOCN3SJ5GQ,Size: 16 Ounces Color: Mint Chocolate,3,1


In [2]:
# Find potential date columns
date_cols = [
    col for col in df.columns
    if any(key in col.lower()
           for key in ["date", "time", "created", "posted"])
]

print("Potential date columns:", date_cols)

for col in date_cols:
    print(f"\n=== {col} ===")
    print(df[col].head(10).to_string(index=False))

Potential date columns: ['date']

=== date ===
Sat Jul 05 2025
Fri Jun 06 2025
Thu Jun 26 2025
Fri Jun 13 2025
Tue Jul 01 2025
Thu Jun 26 2025
Wed Jun 25 2025
Wed Jul 02 2025
Sun Jun 08 2025
Fri Jun 27 2025


In [3]:
# Load the full reviews dataset
df = pd.read_csv(url, low_memory=False)

print("Total reviews:", len(df))
print("Columns:", df.columns.tolist())

if date_cols:
    date_col = date_cols[0]

    dates = pd.to_datetime(
        df[date_col],
        errors="coerce",
        utc=True
    )

    print("\nEarliest date:", dates.min())
    print("Latest date:", dates.max())

    print("\nReviews by year:")
    display(
        dates.dt.year
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis("Year")
        .reset_index(name="Review Count")
    )

    print("\n2025 reviews:", (dates.dt.year == 2025).sum())
    print("2026 reviews:", (dates.dt.year == 2026).sum())
else:
    print("No date column detected.")

Total reviews: 255966
Columns: ['id', 'asin', 'url', 'body', 'date', 'title', 'images', 'videos', 'stars', 'user_id', 'variation', 'found_helpful', 'verified_purchase']

Earliest date: 1999-07-12 00:00:00+00:00
Latest date: 2025-07-17 00:00:00+00:00

Reviews by year:


,Year,Review Count
0,1999,2
1,2001,5
2,2002,2
3,2003,4
4,2004,1
5,2005,4
6,2006,6
7,2007,14
8,2008,17
9,2009,29



2025 reviews: 196119
2026 reviews: 0


In [4]:
# Check product identifiers
asin_cols = [
    col for col in df.columns
    if "asin" in col.lower()
]

print("ASIN columns:", asin_cols)

if asin_cols:
    for col in asin_cols:
        print(col, "unique values:", df[col].nunique())
else:
    print("No ASIN column found in reviews.csv")

ASIN columns: ['asin']
asin unique values: 2689


In [5]:
# 2025 reviews overview
df["parsed_date"] = pd.to_datetime(
    df["date"], errors="coerce", utc=True
)

df_2025 = df[df["parsed_date"].dt.year == 2025].copy()

print("2025 reviews:", len(df_2025))
print("2025 unique products:", df_2025["asin"].nunique())
print("2025 unique users:", df_2025["user_id"].nunique())

print("\n2025 rating distribution:")
print(df_2025["stars"].value_counts().sort_index())

print("\nReviews per product:")
display(
    df_2025.groupby("asin")
    .size()
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
)

print("\nMissing review text:")
print(df_2025["body"].isna().sum())

2025 reviews: 196119
2025 unique products: 2661
2025 unique users: 167900

2025 rating distribution:
stars
1      1858
2      2340
3      8731
4     28767
5    154423
Name: count, dtype: int64

Reviews per product:


count    2661.000000
mean       73.701240
std        28.981129
min         1.000000
50%        86.000000
75%        99.000000
90%       100.000000
95%       100.000000
99%       100.000000
max       100.000000
dtype: float64


Missing review text:
200


In [6]:
import pandas as pd

products_url = (
    "https://huggingface.co/datasets/"
    "datahiveai/Amazon-Reviews-Dataset/resolve/main/products.csv"
)

products = pd.read_csv(products_url, nrows=1000)

print("Shape:", products.shape)
print("\nColumns:")
print(products.columns.tolist())

display(products.head(5))

Shape: (1000, 10)

Columns:
['url', 'asin', 'main_image', 'alt_images', 'title', 'breadcrumbs', 'rating', 'number_of_ratings', 'description', 'characteristics']


,url,asin,main_image,alt_images,title,breadcrumbs,rating,number_of_ratings,description,characteristics
0,https://www.amazon.com/dp/B085DVHQ57,B085DVHQ57,https://m.media-amazon.com/images/I/71Cfld2aMp...,"[""https://m.media-amazon.com/images/I/31ZQ1IgK...",Owala FreeSip Insulated Stainless Steel Water ...,Home & Kitchen -> Kitchen & Dining -> Storage ...,4.6,78303,From the manufacturer Owala FreeSip Insulated ...,"{""0"":{""Brand Name"":""Owala"",""Age Range Descript..."
1,https://www.amazon.com/dp/B0D2Z1PJWT,B0D2Z1PJWT,https://m.media-amazon.com/images/I/61cPXp+NJ2...,"[""https://m.media-amazon.com/images/I/31rxHopx...",STANLEY Quencher H2.0 FlowState Stainless Steel,Home & Kitchen -> Kitchen & Dining -> Storage ...,4.7,89843,From the brand Stanley Visit the store Previou...,"{""0"":{""Color"":""Passion Pink"",""Product Style"":""..."
2,https://www.amazon.com/dp/B086KSDTQ4,B086KSDTQ4,https://m.media-amazon.com/images/I/81ZBOce1Ww...,"[""https://m.media-amazon.com/images/I/410IPcjL...","Hanes Men's Boxer Briefs, Cool Dri Moisture-Wi...","Clothing, Shoes & Jewelry -> Men -> Clothing -...",4.5,166826,From the manufacturer Compare Hanes Boxer Brie...,{}
3,https://www.amazon.com/dp/B09Q3MYDQH,B09Q3MYDQH,https://m.media-amazon.com/images/I/61GzQfBa+l...,"[""https://m.media-amazon.com/images/I/411n-mUo...",Water Shoes for Women Men Quick-Dry Aqua Socks...,"Clothing, Shoes & Jewelry -> Women -> Shoes ->...",4.4,20614,From the brand Previous page More Water Shoes ...,{}
4,https://www.amazon.com/dp/B06X9NQ8GX,B06X9NQ8GX,https://m.media-amazon.com/images/I/71eiII6MS-...,"[""https://m.media-amazon.com/images/I/41LRteDP...",Amazon Basics Digital Kitchen Scale with LCD D...,Home & Kitchen -> Kitchen & Dining -> Kitchen ...,4.7,103279,From the brand Previous page Daily Living Visi...,"{""0"":{""Special Features"":""Large Display, Strai..."


In [7]:
# Extract main product category from breadcrumbs
products["main_category"] = (
    products["breadcrumbs"]
    .fillna("Unknown")
    .astype(str)
    .str.split(" -> ")
    .str[0]
    .str.strip()
)

print("Total products:", len(products))

print("\nProduct categories:")
display(
    products["main_category"]
    .value_counts(dropna=False)
    .head(30)
)

Total products: 1000

Product categories:


main_category
Toys & Games                 86
Unknown                      71
Collectibles & Fine Art      68
Electronics                  51
Home & Kitchen               46
Cell Phones & Accessories    45
Clothing, Shoes & Jewelry    41
Beauty & Personal Care       38
Appliances                   35
Sports & Outdoors            33
Grocery & Gourmet Food       32
Health & Household           32
Baby Products                31
Office Products              30
CDs & Vinyl                  30
Movies & TV                  30
Musical Instruments          30
Software                     30
Video Games                  29
Pet Supplies                 29
Gift Cards                   29
Automotive                   28
Arts, Crafts & Sewing        26
Patio, Lawn & Garden         26
Industrial & Scientific      24
Tools & Home Improvement     23
Books                         5
Kindle Store                  5
Safety & Security             4
Electrónica                   2
Name: count, dtype: int64

In [8]:
# Load the complete product dataset
products = pd.read_csv(products_url, low_memory=False)

products["main_category"] = (
    products["breadcrumbs"]
    .fillna("Unknown")
    .astype(str)
    .str.split(" -> ")
    .str[0]
    .str.strip()
)

print("Total products:", len(products))
print("Unique ASINs:", products["asin"].nunique())

display(
    products["main_category"]
    .value_counts()
    .head(30)
)

Total products: 2727
Unique ASINs: 2727


main_category
Toys & Games                           508
Arts, Crafts & Sewing                  356
Baby Products                          284
Beauty & Personal Care                 266
Patio, Lawn & Garden                   164
Home & Kitchen                         136
Health & Household                     118
Collectibles & Fine Art                105
Unknown                                 75
Electronics                             73
Office Products                         67
Tools & Home Improvement                66
Sports & Outdoors                       49
Cell Phones & Accessories               45
Clothing, Shoes & Jewelry               43
Appliances                              42
Grocery & Gourmet Food                  32
CDs & Vinyl                             30
Movies & TV                             30
Musical Instruments                     30
Software                                30
Automotive                              29
Video Games                             

In [9]:
reviews_2025 = df_2025.merge(
    products[["asin", "main_category"]].drop_duplicates("asin"),
    on="asin",
    how="left",
    validate="many_to_one"
)

category_stats = (
    reviews_2025
    .groupby("main_category", dropna=False)
    .agg(
        review_count=("id", "count"),
        product_count=("asin", "nunique")
    )
    .sort_values("review_count", ascending=False)
)

category_stats["percentage"] = (
    category_stats["review_count"]
    / len(reviews_2025) * 100
).round(2)

display(category_stats.head(30))

print(
    "Matched reviews:",
    reviews_2025["main_category"].notna().sum(),
    "/",
    len(reviews_2025)
)

,review_count,product_count,percentage
main_category,,,
Toys & Games,33924,508,17.30
Beauty & Personal Care,24019,266,12.25
Baby Products,21674,284,11.05
"Arts, Crafts & Sewing",19525,347,9.96
"Patio, Lawn & Garden",12885,163,6.57
Home & Kitchen,10728,136,5.47
Health & Household,9635,118,4.91
Unknown,6475,72,3.30
Electronics,5651,73,2.88


Matched reviews: 196119 / 196119


In [10]:
beauty_2025 = reviews_2025[
    reviews_2025["main_category"] == "Beauty & Personal Care"
].copy()

print("Beauty reviews:", len(beauty_2025))
print("Unique ASINs:", beauty_2025["asin"].nunique())

display(
    beauty_2025.groupby("asin")
    .agg(
        review_count=("id", "count"),
        avg_rating=("stars", "mean"),
        earliest=("parsed_date", "min"),
        latest=("parsed_date", "max")
    )
    .sort_values("review_count", ascending=False)
    .head(20)
)

Beauty reviews: 24019
Unique ASINs: 266


,review_count,avg_rating,earliest,latest
asin,,,,
B0F1G3VSLZ,100,3.35,2025-05-08 00:00:00+00:00,2025-07-15 00:00:00+00:00
B01KYRUBT8,100,4.59,2025-04-08 00:00:00+00:00,2025-07-15 00:00:00+00:00
B074PTZCNX,100,4.83,2025-03-28 00:00:00+00:00,2025-07-13 00:00:00+00:00
B071914GGL,100,4.88,2025-05-23 00:00:00+00:00,2025-07-07 00:00:00+00:00
B0CLX1HQ17,100,4.75,2025-01-07 00:00:00+00:00,2025-07-10 00:00:00+00:00
B01N7T7JKJ,100,4.86,2025-01-14 00:00:00+00:00,2025-07-12 00:00:00+00:00
B01N34XW93,100,4.90,2025-02-12 00:00:00+00:00,2025-07-12 00:00:00+00:00
B01MYEZPC8,100,4.92,2025-03-28 00:00:00+00:00,2025-07-12 00:00:00+00:00
B01MDTVZTZ,100,4.86,2025-05-16 00:00:00+00:00,2025-07-08 00:00:00+00:00
